# 4.4 RandomForest Model

This notebook trains an XGBoost model to predict charging station suitability scores for cities.

## Objectives:
1. Load processed data
2. Train XGBoost model
3. Evaluate model performance
4. Analyze feature importance
5. Save trained model


In [12]:
# Train Random Forest Model - self-contained
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

print("Loading processed arrays...")
# Paths are relative to notebooks/model_training/
X_train = np.load('../../data/processed/X_train_clean.npy')
y_train = np.load('../../data/processed/y_train_clean.npy')
X_test = np.load('../../data/processed/X_test_clean.npy')
y_test = np.load('../../data/processed/y_test_clean.npy')
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

# Ensure 1D targets
y_train = np.array(y_train).ravel()
y_test = np.array(y_test).ravel()

# Check for NaNs in targets and remove corresponding samples
train_nans = np.isnan(y_train).sum()
test_nans = np.isnan(y_test).sum()
print(f"y_train NaNs: {train_nans}, y_test NaNs: {test_nans}")

if train_nans > 0:
    print("Removing samples with NaN targets from training set...")
    mask = ~np.isnan(y_train)
    X_train = X_train[mask]
    y_train = y_train[mask]
    print(f"New training shape: {X_train.shape}, {y_train.shape}")

if test_nans > 0:
    print("Removing samples with NaN targets from test set...")
    mask = ~np.isnan(y_test)
    X_test = X_test[mask]
    y_test = y_test[mask]
    print(f"New test shape: {X_test.shape}, {y_test.shape}")

# Initialize and train the model
print("Training Random Forest Model...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Make predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

print("✓ Model trained successfully")

# Calculate metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("\nModel Performance:")
print("=" * 30)
print(f"Training MSE: {train_mse:.4f}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Training R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Training MAE: {train_mae:.4f}")
print(f"Test MAE: {test_mae:.4f}")


Loading processed arrays...
X_train: (319, 11), y_train: (319,)
X_test:  (87, 11), y_test:  (87,)
y_train NaNs: 0, y_test NaNs: 0
Training Random Forest Model...
✓ Model trained successfully

Model Performance:
Training MSE: 229872.9508
Test MSE: 8747.0506
Training R²: 0.7780
Test R²: 0.9742
Training MAE: 43.1350
Test MAE: 20.5873
✓ Model trained successfully

Model Performance:
Training MSE: 229872.9508
Test MSE: 8747.0506
Training R²: 0.7780
Test R²: 0.9742
Training MAE: 43.1350
Test MAE: 20.5873


In [13]:
# Save Model and Results
import joblib, json, os
print("Saving Model and Results...")
# Save the trained model
os.makedirs('../../models', exist_ok=True)
joblib.dump(rf_model, '../../models/random_forest.pkl')
print("✓ Random Forest model saved")

# Save feature importance if feature_columns available
try:
    import pickle, pandas as pd
    feature_columns_p = '../../data/processed/feature_columns.pkl'
    if os.path.exists(feature_columns_p):
        with open(feature_columns_p, 'rb') as f:
            feature_columns = pickle.load(f)
        fi = pd.DataFrame({
            'Feature': feature_columns,
            'Importance': rf_model.feature_importances_
        }).sort_values('Importance', ascending=False)
        fi.to_csv('../../data/processed/rf_feature_importance.csv', index=False)
        print("✓ Feature importance saved")
    else:
        print("feature_columns.pkl not found; skipping feature importance output")
except Exception as e:
    print('Could not save feature importance:', e)

# Save model performance metrics
performance_metrics = {
    'model_name': 'RandomForest',
    'n_estimators': 100,
    'max_depth': 10,
    'train_mse': float(train_mse),
    'test_mse': float(test_mse),
    'train_r2': float(train_r2),
    'test_r2': float(test_r2),
    'train_mae': float(train_mae),
    'test_mae': float(test_mae),
    'n_features': int(X_train.shape[1]) if X_train.ndim == 2 else None,
    'n_train_samples': int(len(X_train)),
    'n_test_samples': int(len(X_test))
}
with open('../../data/processed/rf_performance_metrics.json', 'w') as f:
    json.dump(performance_metrics, f, indent=2)
print("✓ Performance metrics saved")
print("\nRandom Forest model training completed successfully!")

Saving Model and Results...
✓ Random Forest model saved
✓ Feature importance saved
✓ Performance metrics saved

Random Forest model training completed successfully!
